# Employees Performance Analysis Notebook
## 1) Libraries importation 

In [ ]:
# Importing of differents libraries
import pandas as pd 
import matplotlib.pyplot as plt 
import scipy.stats as stats
import seaborn as sns
from scipy.stats import f_oneway
from scipy.stats import kruskal
from scipy.stats import shapiro

## 2) Data set loading  and exploration

In [ ]:
# Loading of the dataset 
df = pd.read_csv(r"C:\Users\Espadon\Desktop\PortfolioProjects-main\Extended_Employee_Performance_and_Productivity_Data.csv", encoding='ISO-8859-1')
df.head(10)

In [ ]:
# data exploration 
print(df.shape)
print(df.columns)
print(df.dtypes)
df.head()

## 3) Data cleaning and Preprocessing

In [ ]:
# Checking for duplicated values
print("Number of duplicated rows:", df.duplicated().sum())

# Dropping duplicates
df = df.drop_duplicates()

# Checking for missing values
print("\nMissing values per column:", df.isna().sum())

# Convert date column
df["Hire_Date"] = pd.to_datetime(df["Hire_Date"], errors='coerce')

# set "Employee_ID" as index

df = df.set_index("Employee_ID")

# Standardize column names
df.columns = df.columns.str.lower()

# transform performance level into news variables 
def performance_level(score):
    if score <= 2:
        return 'Low'
    elif score == 3:
        return 'Medium'
    else:
        return 'High'

df['Performance_Level'] = df['performance_score'].apply(performance_level)

# Categorize "remote_work_frequency"
df['remote_category'] = df['remote_work_frequency'].map({
    0: 'Never',
    25: 'Rarely',
    50: 'Sometimes',
    75: 'Often',
    100: 'Always'
})

## 4) Exploratory Data Analysis(EDA)
### Demographic

In [ ]:
# Total number of employees 
print("Total Employees:", df.shape[0])

# Employees distribution  by gender
employees_by_gender = df["gender"].value_counts(normalize= True)
print("Employees distribution  by gender\n", employees_by_gender)

# Employees distribution  by educational level
employees_by_education = df['education_level'].value_counts(normalize= True)
print(" Employees distribution  by Marital status\n", employees_by_education )

### Basic analysis 

In [ ]:
# Employees distribution  by department 
employees_by_department = df["department"].value_counts(normalize= True)
print("Employees distribution  by department \n", employees_by_department)

# Employees distribution  by job title
employees_by_job = df['job_title'].value_counts(normalize= True)
print(" Employees distribution  by job title\n", employees_by_job )

# employee distribution by working style 
employees_by_work = df['remote_category'].value_counts(normalize= True)
print(" employee distribution by working stylee\n", employees_by_work )


### Descriptive analysis 

In [ ]:
df.describe()

## 5) Performance analysis 
### Exploratory analysis 

In [ ]:
# Performance by gender
performance_by_gender = pd.crosstab(df['gender'], df['Performance_Level']) 
print("performance by department\n", performance_by_gender)

#Performance by department 
performance_by_department = pd.crosstab(df['department'], df['Performance_Level'])
print("Average performance by department\n", performance_by_department)

# Performance by job title
performance_by_job_title = pd.crosstab(df['job_title'], df['Performance_Level'])
print("Average performance by job title\n", performance_by_job_title)

# Performance by education level
performance_by_education = pd.crosstab(df['education_level'], df['Performance_Level'])
print("Average performance by education level\n", performance_by_education)

# performance by remote category
performance_by_remote_category = pd.crosstab(df['remote_category'], df['Performance_Level'])
print("Performance by remote category\n", performance_by_remote_category)

# Performance by resignation 
performance_by_resignation = pd.crosstab(df['resigned'], df['Performance_Level'])
print("Performance by remote category\n", performance_by_resignation)


### Chisquare analysis 

In [ ]:
# Chi square analysis of the performance related to other varaiables
from scipy.stats import chi2_contingency
# test if the performance_level  is function of other categorical variable 
cat_col = df.select_dtypes(include = ["object", "bool"]).columns
final_cat_col = cat_col.drop(["Performance_Level"])
target = "Performance_Level"
df_clean = df[[*final_cat_col, target]]

chi2_results = []
for col in final_cat_col:
    contingency_table = pd.crosstab(df_clean[col], df_clean[target])
    if contingency_table.shape[0] >1 and contingency_table.shape[1]>1:
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        chi2_results.append({
            "variable": col,
            "Chi2 Statistic": chi2,
            "p-value": p,
            "Degree of Freedom": dof})
result = pd.DataFrame(chi2_results).sort_values(by ="p-value")

print(result)

### Performance analysis with other quantitative variables 

In [ ]:
# Select numerical variables
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
# Exclude target if accidentally in num_cols
num_cols = num_cols.drop(['performance_score',"remote_work_frequency"])


In [ ]:
# test for normality  
for col in num_cols:
    stat, p_value = shapiro(df[col])
    print(f"{col}: p-value = {p_value:.4f} (Normal: {'Yes' if p_value > 0.05 else 'No'})")

In [ ]:
# Boxplots for visual comparison ===
print("=== Boxplots ===")
for col in num_cols:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x='Performance_Level', y=col, data=df)
    plt.title(f"{col} by Performance Level")
    plt.tight_layout()
    plt.show()

In [ ]:
# Run Kruskal-Wallis for each numerical variable
results = {}
for col in num_cols:
    groups = [df[df['Performance_Level'] == dept][col] for dept in df["Performance_Level"].unique()]
    stat, p_value = kruskal(*groups)
    results[col] = {'H-statistic': stat, 'p-value': p_value}

# Convert results to a DataFrame for better readability
import pandas as pd
results_df = pd.DataFrame(results).T
results_df['Significant? (p < 0.05)'] = results_df['p-value'] < 0.05
print(results_df.sort_values(by='p-value'))

In [ ]:
df.head(10)

## 5) Satisfaction  analysis 
### Exploratory analysis 

In [ ]:
# Boxplots for visual comparison 
print("=== Boxplots ===")
for col in cat_col:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=col, y='employee_satisfaction_score', data=df)
    plt.title(f"{col} by satisfaction Level")
    plt.tight_layout()
    plt.show()

In [ ]:
# Satisfaction by gender
satisfaction_by_gender = df.groupby("gender")["employee_satisfaction_score"].mean()
print("Satisfaction by gendert\n", satisfaction_by_gender)